In [47]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import END,START
from langgraph.graph.state import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
import os
from dotenv import load_dotenv
load_dotenv()

True

In [48]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [49]:
from langchain.chat_models import init_chat_model

llm=init_chat_model("groq:llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7973226fae40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7973226f7e90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [50]:
class State(TypedDict):
    message:Annotated[list[BaseMessage],add_messages]

In [52]:
## Graph with tool call
from langchain_core.tools import tool

@tool
def add(a:float,b:float):
    """Add two numbers"""
    return a+b
tools=[add]

tool_node = ToolNode([add])
llm_with_tool=llm.bind_tools([add])

def call_llm_model(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

In [ ]:
## Stategraph
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condtion

## Node definition
def call_llm_model(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}
## Graph
builder=StateGraph(State)
builder.add_node("tool_calling_llm",call_llm_model)
builder.add_node("tools",ToolNode(tools))

## Add Edges
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condtion
)
builder.add_edge("tools",END)

## Compile the graph
graph=builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

ImportError: cannot import name 'tools_condtion' from 'langgraph.prebuilt' (/home/frank/lang_graph/.venv/lib/python3.12/site-packages/langgraph/prebuilt/__init__.py)

In [57]:
response=graph.invoke({"messages":"What is the recent ai news?"})

KeyError: 'messages'